In [1]:
import pandas as pd
import numpy as np
import joblib
import time
import sys

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

sys.path.insert(0, str(Path("..").resolve()))

from src.preprocessing import create_preprocessor

RANDOM_STATE = 42

In [2]:
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
RESULT_DIR = Path("../reports/results")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Directories configured successfully!")

Directories configured successfully!


In [3]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")

y_train = pd.read_csv(
    DATA_DIR / "y_train.csv"
)["label"]

X_val = pd.read_csv(DATA_DIR / "X_val.csv")

y_val = pd.read_csv(
    DATA_DIR / "y_val.csv"
)["label"]

print("Training Features:", X_train.shape)
print("Training Target:", y_train.shape)

print("\nValidation Features:", X_val.shape)
print("Validation Target:", y_val.shape)

Training Features: (140274, 42)
Training Target: (140274,)

Validation Features: (35067, 42)
Validation Target: (35067,)


In [4]:
print("Input Features:", X_train.shape[1])

print("\nTarget Distribution:")
print(y_train.value_counts())

print("\nMissing Values:",
      X_train.isnull().sum().sum())

assert len(X_train) == len(y_train)

assert X_train.columns.equals(X_val.columns)

assert set(y_train.unique()) == {0, 1}

print("\nData verification successful!")

Input Features: 42

Target Distribution:
label
1    95474
0    44800
Name: count, dtype: int64

Missing Values: 0

Data verification successful!


In [5]:
lr_preprocessor = create_preprocessor(
    X_train, scale=True
)

dt_preprocessor = create_preprocessor(
    X_train, scale=False
)

rf_preprocessor = create_preprocessor(
    X_train, scale=False
)

print("All preprocessing pipelines created!")

All preprocessing pipelines created!


In [6]:
logistic_model = Pipeline([
    ("preprocessor", lr_preprocessor),

    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

print("Logistic Regression model created!")

Logistic Regression model created!


In [7]:
decision_tree_model = Pipeline([
    ("preprocessor", dt_preprocessor),

    ("classifier", DecisionTreeClassifier(
        criterion="gini",
        random_state=RANDOM_STATE
    ))
])

print("Decision Tree model created!")

Decision Tree model created!


In [8]:
random_forest_model = Pipeline([
    ("preprocessor", rf_preprocessor),

    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

print("Random Forest model created!")

Random Forest model created!


In [9]:
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model
}

print("Total Models:", len(models))

for name in models:
    print(name)

Total Models: 3
Logistic Regression
Decision Tree
Random Forest


In [10]:
training_results = []

for name, model in models.items():

    print(f"\nTraining {name}...")

    start_time = time.perf_counter()

    model.fit(X_train, y_train)

    end_time = time.perf_counter()

    training_time = end_time - start_time

    training_results.append({
        "Model": name,
        "Training Time (Seconds)": round(training_time, 2),
        "Status": "Successfully Trained"
    })

    print(f"{name} trained successfully!")

    print(f"Training Time: {training_time:.2f} seconds")

print("\nAll models trained successfully!")


Training Logistic Regression...
Logistic Regression trained successfully!
Training Time: 13.32 seconds

Training Decision Tree...
Decision Tree trained successfully!
Training Time: 19.54 seconds

Training Random Forest...
Random Forest trained successfully!
Training Time: 137.91 seconds

All models trained successfully!


In [11]:
training_summary = pd.DataFrame(training_results)

display(training_summary)

,Model,Training Time (Seconds),Status
0,Logistic Regression,13.32,Successfully Trained
1,Decision Tree,19.54,Successfully Trained
2,Random Forest,137.91,Successfully Trained


In [15]:
sample_data = X_val.head(10)

prediction_summary = pd.DataFrame({
    "Actual Label": y_val.head(10).values
})

for name, model in models.items():

    predictions = model.predict(sample_data)

    prediction_summary[name] = predictions

display(prediction_summary)

prediction_summary.replace({
    0: "Normal",
    1: "Malicious"
})

,Actual Label,Logistic Regression,Decision Tree,Random Forest
0,0,0,0,0
1,0,0,0,0
2,0,1,1,1
3,0,1,1,1
4,0,1,0,0
5,0,0,0,1
6,0,1,1,1
7,0,1,1,1
8,0,1,0,1
9,0,1,1,1


,Actual Label,Logistic Regression,Decision Tree,Random Forest
0,Normal,Normal,Normal,Normal
1,Normal,Normal,Normal,Normal
2,Normal,Malicious,Malicious,Malicious
3,Normal,Malicious,Malicious,Malicious
4,Normal,Malicious,Normal,Normal
5,Normal,Normal,Normal,Malicious
6,Normal,Malicious,Malicious,Malicious
7,Normal,Malicious,Malicious,Malicious
8,Normal,Malicious,Normal,Malicious
9,Normal,Malicious,Malicious,Malicious


In [16]:
model_filenames = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "Random Forest": "random_forest.pkl"
}

for name, model in models.items():

    file_path = MODEL_DIR / model_filenames[name]

    joblib.dump(model, file_path, compress=3)

    print(f"{name} saved successfully!")

print("\nAll models saved successfully!")

Logistic Regression saved successfully!
Decision Tree saved successfully!
Random Forest saved successfully!

All models saved successfully!


In [17]:
for name, filename in model_filenames.items():

    loaded_model = joblib.load(
        MODEL_DIR / filename
    )

    original_predictions = models[name].predict(
        sample_data
    )

    loaded_predictions = loaded_model.predict(
        sample_data
    )

    assert np.array_equal(
        original_predictions,
        loaded_predictions
    )

    print(f"{name}: Saved model verified successfully!")

Logistic Regression: Saved model verified successfully!
Decision Tree: Saved model verified successfully!
Random Forest: Saved model verified successfully!


In [18]:
for name, filename in model_filenames.items():

    file_path = MODEL_DIR / filename

    size_mb = file_path.stat().st_size / (1024 ** 2)

    print(f"{name}: {size_mb:.2f} MB")

Logistic Regression: 0.01 MB
Decision Tree: 0.13 MB
Random Forest: 16.04 MB


In [19]:
training_summary.to_csv(
    RESULT_DIR / "training_summary.csv",
    index=False
)

print("Training summary saved successfully!")

Training summary saved successfully!
